In [2]:
!apt-get update
!apt-get install -y zstd

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [93.4 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,615 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,002 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-up

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Ollama server started")

Ollama server started


In [5]:
!ollama pull llama3

In [6]:
!pip install ollama pandas tqdm

In [7]:
import ollama

response = ollama.chat(
    model="llama3",
    messages=[
        {"role": "user", "content": "Hello. Reply with one sentence."}
    ]
)

print(response["message"]["content"])

It's nice to meet you!


In [8]:
import random
import pandas as pd
from tqdm import tqdm
import ollama
from google.colab import files

In [9]:
model_name = "llama3"

In [12]:
uploaded = files.upload()

Saving human_negotiations.csv to human_negotiations.csv


In [13]:
human_df = pd.read_csv("human_negotiations.csv")

print(human_df.shape)
human_df.head()

(4993, 8)


,dialogue_id,split,turn_id,role,text,act,issue,value
0,0,train,0,Candidate,Hello. I would like to discuss the issues of m...,Greet,NaN,True
1,0,train,1,Candidate,I would like a position of project manager,Offer,Job Description,Project Manager
2,0,train,2,Employer,We do not have any project manager positions o...,Reject,Job Description,Project Manager
3,0,train,3,Candidate,I want a position of project manager,Offer,Job Description,Project Manager
4,0,train,4,Employer,I do have programmer positions open with a str...,Offer,Job Description,Programmer


In [14]:
print("Number of dialogues:", human_df["dialogue_id"].nunique())
print(human_df.columns)

Number of dialogues: 105
Index(['dialogue_id', 'split', 'turn_id', 'role', 'text', 'act', 'issue',
       'value'],
      dtype='object')


In [15]:
def generate_response(system_prompt, user_message):
    response = ollama.chat(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_message
            }
        ]
    )

    return response["message"]["content"].strip()

In [16]:
test_response = generate_response(
    "You are a concise assistant.",
    "Say hello in one sentence."
)

print(test_response)

Hello!


In [17]:
NEGOTIATION_ACTS = ["Offer", "Reject", "Query", "Accept"]

def get_first_negotiation_act(human_df, dialogue_id):
    dialogue = human_df[
        (human_df["dialogue_id"].astype(str) == str(dialogue_id)) &
        (human_df["act"].isin(NEGOTIATION_ACTS))
    ].sort_values("turn_id")

    if dialogue.empty:
        return None

    return dialogue.iloc[0]

In [18]:
#test
example_dialogue_id = human_df["dialogue_id"].astype(str).unique()[0]

first_act = get_first_negotiation_act(
    human_df,
    example_dialogue_id
)

first_act

,1
dialogue_id,0
split,train
turn_id,1
role,Candidate
text,I would like a position of project manager
act,Offer
issue,Job Description
value,Project Manager


In [19]:
def build_agent_prompt(role):
    return f"""
You are the {role} in a job contract negotiation.

You are continuing a negotiation that started from a real human negotiation act.

Your goal:
- negotiate realistically according to your role
- make offers, counteroffers, rejections, questions, or acceptances when appropriate
- avoid repeating the same point
- try to reach an agreement if possible
- if the negotiation is clearly stuck, end it

Use one of these dialogue acts:
Offer, Accept, Reject, Query, Quit, Other

Rules:
- If you accept the other party's proposal, write ACT: Accept and DECISION: accept.
- If no compromise is possible, write ACT: Quit and DECISION: quit.
- If you are still negotiating, write DECISION: continue.
- Keep the message short and realistic.
- Do not simulate both speakers. Reply only as your assigned role.

Reply ONLY in this format:

MESSAGE: your short negotiation message
ACT: Offer / Accept / Reject / Query / Quit / Other
ISSUE: negotiated issue or NONE
VALUE: proposed value or NONE
DECISION: continue / accept / quit
"""

In [20]:
def parse_llm_act_response(text):
    result = {
        "message": None,
        "act": None,
        "issue": None,
        "value": None,
        "decision": None
    }

    for line in text.splitlines():
        line = line.strip()

        if line.startswith("MESSAGE:"):
            result["message"] = line.replace("MESSAGE:", "").strip()

        elif line.startswith("ACT:"):
            result["act"] = line.replace("ACT:", "").strip()

        elif line.startswith("ISSUE:"):
            value = line.replace("ISSUE:", "").strip()
            result["issue"] = None if value.upper() == "NONE" else value

        elif line.startswith("VALUE:"):
            value = line.replace("VALUE:", "").strip()
            result["value"] = None if value.upper() == "NONE" else value

        elif line.startswith("DECISION:"):
            result["decision"] = line.replace("DECISION:", "").strip().lower()

    return result

In [21]:
def build_seed_message(first_act):
    return f"""
The following is the first substantive negotiation act from a human job negotiation.

Speaker: {first_act["role"]}
Utterance: {first_act["text"]}
Dialogue act: {first_act["act"]}
Issue: {first_act["issue"]}
Value: {first_act["value"]}

Continue the negotiation from this point.
"""

In [22]:
def get_next_speaker(role):
    if role == "Candidate":
        return "Employer"
    elif role == "Employer":
        return "Candidate"
    else:
        return "Employer"

In [23]:
def run_human_seeded_simulation(human_df, dialogue_id, max_cap=20):
    first_act = get_first_negotiation_act(human_df, dialogue_id)

    if first_act is None:
        return [], {
            "dialogue_id": dialogue_id,
            "outcome": "NoSeed",
            "n_turns": 0,
            "human_original_turns": 0,
            "seed_role": None,
            "seed_act": None,
            "seed_issue": None,
            "seed_value": None
        }

    human_dialogue = human_df[
        human_df["dialogue_id"].astype(str) == str(dialogue_id)
    ]

    human_length = human_dialogue["turn_id"].nunique()
    max_turns = min(human_length, max_cap)

    seed_message = build_seed_message(first_act)

    conversation_log = []

    conversation_log.append({
        "dialogue_id": dialogue_id,
        "turn": 0,
        "speaker": first_act["role"],
        "text": first_act["text"],
        "act": first_act["act"],
        "issue": first_act["issue"],
        "value": first_act["value"],
        "decision": "seed",
        "is_seed": True
    })

    current_message = seed_message
    current_speaker = get_next_speaker(first_act["role"])

    outcome = None

    for turn in range(1, max_turns + 1):

        system_prompt = build_agent_prompt(current_speaker)

        llm_text = generate_response(
            system_prompt,
            current_message
        )

        parsed = parse_llm_act_response(llm_text)

        conversation_log.append({
            "dialogue_id": dialogue_id,
            "turn": turn,
            "speaker": current_speaker,
            "text": llm_text,
            "act": parsed["act"],
            "issue": parsed["issue"],
            "value": parsed["value"],
            "decision": parsed["decision"],
            "is_seed": False
        })

        if parsed["decision"] == "accept" or parsed["act"] == "Accept":
            outcome = "Agreement"
            break

        if parsed["decision"] == "quit" or parsed["act"] == "Quit":
            outcome = "Failure"
            break

        recent_context = conversation_log[-6:]

        context_text = ""
        for row in recent_context:
            context_text += f'{row["speaker"]}: {row["text"]}\n'

        next_speaker = get_next_speaker(current_speaker)

        current_message = f"""
Continue the following job negotiation.

Recent conversation:
{context_text}

Now reply as {next_speaker}.
"""

        current_speaker = next_speaker

    if outcome is None:
        recent_non_seed = [
            row for row in conversation_log[-6:]
            if row["is_seed"] == False
        ]

        recent_acts = [
            str(row["act"]).lower()
            for row in recent_non_seed
            if row["act"] is not None
        ]

        recent_decisions = [
            str(row["decision"]).lower()
            for row in recent_non_seed
            if row["decision"] is not None
        ]

        recent_texts = [
            str(row["text"]).lower()
            for row in recent_non_seed
            if row["text"] is not None
        ]

        recent_text = " ".join(recent_texts)

        impasse_keywords = [
            "cannot accept",
            "can't accept",
            "not acceptable",
            "too low",
            "too high",
            "below my minimum",
            "above our maximum",
            "no agreement",
            "unable to agree",
            "end the negotiation",
            "walk away"
        ]

        if (
            "reject" in recent_acts
            or recent_decisions.count("continue") >= 4
            or any(keyword in recent_text for keyword in impasse_keywords)
        ):
            outcome = "Impasse"
        else:
            outcome = "Timeout"

    outcome_row = {
        "dialogue_id": dialogue_id,
        "outcome": outcome,
        "n_turns": len(conversation_log),
        "human_original_turns": human_length,
        "seed_role": first_act["role"],
        "seed_act": first_act["act"],
        "seed_issue": first_act["issue"],
        "seed_value": first_act["value"]
    }

    return conversation_log, outcome_row

In [24]:
test_dialogues = human_df["dialogue_id"].astype(str).unique()[:3]

test_turns = []
test_outcomes = []

for dialogue_id in test_dialogues:
    print("Running test dialogue:", dialogue_id)

    log, outcome = run_human_seeded_simulation(
        human_df,
        dialogue_id,
        max_cap=8
    )

    test_turns.extend(log)
    test_outcomes.append(outcome)

test_turns_df = pd.DataFrame(test_turns)
test_outcomes_df = pd.DataFrame(test_outcomes)

test_outcomes_df

Running test dialogue: 0
Running test dialogue: 1
Running test dialogue: 10


,dialogue_id,outcome,n_turns,human_original_turns,seed_role,seed_act,seed_issue,seed_value
0,0,Impasse,9,55,Candidate,Offer,Job Description,Project Manager
1,1,Impasse,9,27,Candidate,Offer,Job Description,Project Manager
2,10,Impasse,9,63,Candidate,Offer,Job Description,Project Manager


In [25]:
test_turns_df[["dialogue_id", "turn", "speaker", "act", "decision", "text"]].head(30)

,dialogue_id,turn,speaker,act,decision,text
0,0,0,Candidate,Offer,seed,I would like a position of project manager
1,0,1,Employer,Offer,continue,"MESSAGE: Unfortunately, our current opening fo..."
2,0,2,Candidate,Query,continue,"MESSAGE: While I appreciate the offer, I'm con..."
3,0,3,Employer,Offer,continue,MESSAGE: We value your leadership skills and w...
4,0,4,Candidate,Query,continue,MESSAGE: I'm intrigued by the idea of a hybrid...
5,0,5,Employer,Offer,continue,MESSAGE: We're looking for someone with strong...
6,0,6,Candidate,Offer,continue,MESSAGE: I'm glad we're exploring the possibil...
7,0,7,Employer,Offer,continue,MESSAGE: We understand your concerns about com...
8,0,8,Candidate,Offer,continue,MESSAGE: I appreciate your willingness to cons...
9,1,0,Candidate,Offer,seed,I am expecting a position of project manager


In [26]:
random.seed(42)

all_dialogues = human_df["dialogue_id"].astype(str).unique().tolist()

sample_size = 20

sample_dialogues = random.sample(
    all_dialogues,
    min(sample_size, len(all_dialogues))
)

print("Number of sampled dialogues:", len(sample_dialogues))
print(sample_dialogues)

Number of sampled dialogues: 20
['78', '17', '100', '9', '36', '32', '3', '2', '16', '82', '67', '14', '72', '53', '101', '29', '30', '62', '74', '69']


In [27]:
human_seeded_turns = []
human_seeded_outcomes = []

for dialogue_id in tqdm(sample_dialogues):
    log, outcome = run_human_seeded_simulation(
        human_df,
        dialogue_id,
        max_cap=20
    )

    human_seeded_turns.extend(log)
    human_seeded_outcomes.append(outcome)

    # autosave
    pd.DataFrame(human_seeded_turns).to_csv(
        "/content/human_seeded_llm_turns_sample.csv",
        index=False
    )

    pd.DataFrame(human_seeded_outcomes).to_csv(
        "/content/human_seeded_llm_outcomes_sample.csv",
        index=False
    )

human_seeded_turns_df = pd.DataFrame(human_seeded_turns)
human_seeded_outcomes_df = pd.DataFrame(human_seeded_outcomes)

human_seeded_outcomes_df

100%|██████████| 20/20 [17:45<00:00, 53.27s/it]


,dialogue_id,outcome,n_turns,human_original_turns,seed_role,seed_act,seed_issue,seed_value
0,78,Agreement,21,47,Candidate,Offer,Job Description,Project Manager
1,17,Impasse,21,48,Candidate,Offer,Salary,"120,000 USD"
2,100,Agreement,17,43,Candidate,Offer,Job Description,Project Manager
3,9,Impasse,21,23,Candidate,Offer,Salary,"120,000 USD"
4,36,Impasse,19,18,Candidate,Offer,Job Description,Project Manager
5,32,Agreement,21,27,Candidate,Offer,Job Description,Project Manager
6,3,Agreement,17,32,Candidate,Offer,Salary,"120,000 USD"
7,2,Agreement,13,16,Candidate,Offer,Salary,"120,000 USD"
8,16,Impasse,15,14,Candidate,Offer,Job Description,Project Manager
9,82,Impasse,21,29,Candidate,Offer,Job Description,Project Manager


In [28]:
human_seeded_outcomes_df["outcome"].value_counts()

,count
outcome,
Impasse,11
Agreement,9


In [29]:
human_seeded_outcomes_df[["n_turns", "human_original_turns"]].describe()

,n_turns,human_original_turns
count,20.000000,20.000000
mean,18.000000,29.650000
std,3.741657,12.175363
min,10.000000,12.000000
25%,16.500000,20.750000
50%,20.000000,27.000000
75%,21.000000,37.750000
max,21.000000,55.000000


In [30]:
human_seeded_outcomes_df.head()

,dialogue_id,outcome,n_turns,human_original_turns,seed_role,seed_act,seed_issue,seed_value
0,78,Agreement,21,47,Candidate,Offer,Job Description,Project Manager
1,17,Impasse,21,48,Candidate,Offer,Salary,"120,000 USD"
2,100,Agreement,17,43,Candidate,Offer,Job Description,Project Manager
3,9,Impasse,21,23,Candidate,Offer,Salary,"120,000 USD"
4,36,Impasse,19,18,Candidate,Offer,Job Description,Project Manager


In [31]:
human_seeded_turns_df.to_csv(
    "/content/human_seeded_llm_turns_sample.csv",
    index=False
)

human_seeded_outcomes_df.to_csv(
    "/content/human_seeded_llm_outcomes_sample.csv",
    index=False
)

print("Saved!")

Saved!


In [32]:
files.download("/content/human_seeded_llm_turns_sample.csv")
files.download("/content/human_seeded_llm_outcomes_sample.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In the human-seeded simulations, unresolved negotiations were mostly classified as Impasse rather than Failure. This reflects the tendency of LLM agents to avoid explicit termination acts such as Quit, even when the negotiation becomes stuck.